# **Generating Quiz questions**
---

## **Reading the `ENGLISH` transcript file**

In [1]:
# Open and read the text file
with open("artifacts/transcript_01.txt", "r", encoding="utf-8") as file:
    transcript = file.read()

print("Transcript:", transcript)

Transcript: if you clicked on this video you're probably familiar with pcie and in got two slots for storage and expansion but you might be overlooking this which is also an m.2 slot but a little bit different it's often just referred to as the Wi-Fi slot but it can actually do a lot more than that and today I'm going to try putting a few different modules in this often overlooked socket to try and get the most performance out of this PC so stay tuned thank you recently I was working on this HP Elite desk 800 G3 mini and sadly one of the Wi-Fi antennas had completely been ripped off now this wasn't a big deal because I typically use wired ethernet but it was kind of sad just looking at the poor little Wi-Fi card with only one cable connected but this kind of got me thinking about how as much as I love finding the most value in used computers I've possibly been overlooking a great opportunity this wi-fi slot is technically an m.2 e key slot you're probably familiar with m and b keyed m.

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = os.getenv("NVIDIA_API_KEY")
)

completion = client.chat.completions.create(
  model="deepseek-ai/deepseek-v4-pro-0813",
  messages=[{
      "role":"user",
      "content":f"Here is the transcript of a youtube video:{transcript}. Please generate 10 multiple choice questions based on this transcript. Each question should have 4 answer options, and indicate the correct answer."}],
  temperature=1,
  top_p=0.95,
  max_tokens=16384,
  seed=42,
  extra_body={"chat_template_kwargs":{"thinking":False}},
  stream=False
)

print(completion.choices[0].message.content)

Here are 10 multiple choice questions based on the provided video transcript:

**1. What type of slot is the "Wi-Fi slot" technically classified as?**
a) M.2 B key
b) M.2 M key
c) M.2 E key
d) PCIe x16

**Correct Answer: c) M.2 E key**

**2. What is the maximum PCIe support mentioned for the specific M.2 E key slot on the HP EliteDesk 800 G3 mini?**
a) Two lanes of PCIe Gen 3
b) One lane of PCIe Gen 2
c) One lane of PCIe Gen 4
d) Four lanes of PCIe Gen 3

**Correct Answer: b) One lane of PCIe Gen 2**

**3. Why is it important to check a motherboard's specifications before using a different module in the M.2 E key slot?**
a) The slot might be physically too small for other modules.
b) The motherboard manufacturer decides which interfaces are wired up and active.
c) The slot always only supports USB 2.0.
d) Other modules generate too much heat for the slot.

**Correct Answer: b) The motherboard manufacturer decides which interfaces are wired up and active.**

**4. What is the first devic

## **Send it to the LLM but recieve the response in JSON format**

In [3]:
import json

prompt = f"""Here is the transcript of a YouTube video:

{transcript}

Generate exactly 10 multiple choice questions based on this transcript.

Return ONLY valid JSON, with no extra text, no markdown code fences, and no commentary.
The JSON must be a list of exactly 10 objects, each with this exact structure:

{{
  "question": "the question text",
  "options": ["option A", "option B", "option C", "option D"],
  "correct_answer_index": 0
}}

Rules:
- "options" must always have exactly 4 strings.
- "correct_answer_index" must be an integer from 0 to 3, pointing to the correct option in the "options" list.
- Do not include letters like "A)" or "B)" inside the option text itself.
- Base every question strictly on the transcript content — do not invent facts not present in it.
"""

completion = client.chat.completions.create(
    model="deepseek-ai/deepseek-v4-pro-0813",
    messages=[{"role": "user", "content": prompt}],
    temperature=1,
    top_p=0.95,
    max_tokens=16384,
    seed=42,
    extra_body={"chat_template_kwargs": {"thinking": False}},
    stream=False
)

raw_output = completion.choices[0].message.content.strip()
print(raw_output)

```json
[
  {
    "question": "What type of M.2 socket is the Wi-Fi slot on the HP EliteDesk 800 G3 mini?",
    "options": ["E key", "M key", "B key", "A key"],
    "correct_answer_index": 0
  },
  {
    "question": "Which interface can an M.2 E key slot potentially support besides wireless cards?",
    "options": ["PCIe, USB 2.0, and other interfaces", "SATA and USB 3.0", "PCIe x16 and Thunderbolt", "Only USB 3.1"],
    "correct_answer_index": 0
  },
  {
    "question": "What tool did the creator use on Windows to check if the M.2 E key slot supported PCIe?",
    "options": ["Hardware Info 64", "Device Manager", "CPU-Z", "Task Manager"],
    "correct_answer_index": 0
  },
  {
    "question": "What is a potential issue mentioned in the video when trying to use non-Wi-Fi cards in some OEM motherboards?",
    "options": ["Whitelisting of certain cards", "Overheating", "BIOS password", "Missing drivers"],
    "correct_answer_index": 0
  },
  {
    "question": "What speed did the 2.5 gigab

In [4]:
# Some models wrap JSON in ```json ... ``` even when told not to — strip it defensively
if raw_output.startswith("```"):
    raw_output = raw_output.strip("`")
    if raw_output.startswith("json"):
        raw_output = raw_output[4:].strip()

try:
    quiz_data = json.loads(raw_output)
except json.JSONDecodeError as e:
    print("Failed to parse JSON. Raw model output was:\n", raw_output)
    raise e

# Basic validation before saving
assert isinstance(quiz_data, list), "Expected a list of questions"
assert len(quiz_data) == 10, f"Expected 10 questions, got {len(quiz_data)}"
for q in quiz_data:
    assert set(q.keys()) == {"question", "options", "correct_answer_index"}, f"Unexpected keys: {q.keys()}"
    assert len(q["options"]) == 4, f"Expected 4 options, got {len(q['options'])}"
    assert 0 <= q["correct_answer_index"] <= 3, "correct_answer_index out of range"

with open("quiz.json", "w", encoding="utf-8") as f:
    json.dump(quiz_data, f, indent=2, ensure_ascii=False)

print(f"Saved {len(quiz_data)} questions to quiz.json")

Saved 10 questions to quiz.json
